# Dropout

## Learning Objectives
1. Implement inverted dropout from scratch in NumPy with correct rescaling.
2. Compare five dropout rates on the same architecture and measure overfitting gap.
3. Apply MC Dropout for uncertainty estimation in a regression task.
4. Demonstrate variational dropout and scheduled dropout as advanced variants.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Inverted Dropout from Scratch (NumPy)

In [ ]:
# Inverted dropout: zero out fraction p of neurons, then scale by 1/(1-p)
# so expected output magnitude stays constant regardless of dropout rate.

def dropout_forward(x, p=0.5, training=True):
    # Inverted dropout.
    # Args:
    # x: input array
    # p: fraction of neurons set to zero
    # training: if False, return x unchanged (no dropout at inference)
    if not training or p == 0.0:
        return x
    mask = (np.random.rand(*x.shape) > p).astype(float)
    return x * mask / (1.0 - p)   # scale keeps E[output] = E[input]

# Verify: mean of output should match mean of input regardless of dropout rate
x_test = np.random.randn(10000)
print("Dropout verification (mean and std of output across 100 calls):")
for p in [0.0, 0.2, 0.5, 0.8]:
    outputs = [dropout_forward(x_test, p=p, training=True) for _ in range(100)]
    means = [o.mean() for o in outputs]
    print(f"  p={p:.1f}  output_mean={np.mean(means):.4f}  std_of_mean={np.std(means):.4f}")

# Inference: no change
print(f"Inference (p=0.5, training=False): {dropout_forward(np.ones(5), p=0.5, training=False)}")


## Level 2: Dropout Rate Comparison -- 5 Variants (PyTorch)

In [ ]:
# Compare p in {0.0, 0.1, 0.3, 0.5, 0.7}; measure train-val accuracy gap.

def build_dropout_mlp(p):
    return nn.Sequential(
        nn.Linear(20, 128), nn.ReLU(), nn.Dropout(p),
        nn.Linear(128, 128), nn.ReLU(), nn.Dropout(p),
        nn.Linear(128, 2)
    ).to(device)

N_dp = 600
X_dp = torch.randn(N_dp, 20)
y_dp = ((X_dp[:,0] + X_dp[:,1] - X_dp[:,2]) > 0).long()
tr_ds_dp = TensorDataset(X_dp[:480].to(device), y_dp[:480].to(device))
tr_ld_dp = DataLoader(tr_ds_dp, batch_size=32, shuffle=True)

dropout_results = {}
for p in [0.0, 0.1, 0.3, 0.5, 0.7]:
    mdl = build_dropout_mlp(p)
    opt = torch.optim.Adam(mdl.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for epoch in range(100):
        mdl.train()
        for xb, yb in tr_ld_dp:
            opt.zero_grad()
            try:
                crit(mdl(xb), yb).backward()
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print(f"OOM at p={p} -- reduce model size")
                    torch.cuda.empty_cache()
                    continue
                raise
            opt.step()
    mdl.eval()
    with torch.no_grad():
        tr_acc = (mdl(X_dp[:480].to(device)).argmax(1) == y_dp[:480].to(device)).float().mean().item()
        va_acc = (mdl(X_dp[480:].to(device)).argmax(1) == y_dp[480:].to(device)).float().mean().item()
    dropout_results[p] = (tr_acc, va_acc)
    print(f"p={p:.1f}  train={tr_acc:.4f}  val={va_acc:.4f}  gap={tr_acc-va_acc:.4f}")

best_p = max(dropout_results, key=lambda p: dropout_results[p][1])
print(f"Best dropout rate (by val acc): p={best_p}")


## Real-World Example 1: MC Dropout for Uncertainty Estimation

In [ ]:
# MC Dropout: keep dropout ACTIVE at inference, run T forward passes,
# use variance of predictions as uncertainty estimate.

class MCDropoutNet(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_hidden, d_hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(d_hidden, d_out)
        )
    def forward(self, x):
        return self.net(x)

N_mc = 500
X_mc = torch.linspace(-3, 3, N_mc).unsqueeze(1)
y_mc = torch.sin(X_mc) + 0.1 * torch.randn_like(X_mc)
# Training data has a gap in [1, 2] -- model should be uncertain there
tr_mask = (X_mc.squeeze() < 1) | (X_mc.squeeze() > 2)
X_mc_tr = X_mc[tr_mask].to(device)
y_mc_tr = y_mc[tr_mask].to(device)

mc_net = MCDropoutNet(1, 64, 1, p=0.3).to(device)
opt_mc = torch.optim.Adam(mc_net.parameters(), lr=1e-3)
for _ in range(200):
    mc_net.train()
    opt_mc.zero_grad()
    nn.MSELoss()(mc_net(X_mc_tr), y_mc_tr).backward()
    opt_mc.step()

# MC inference: model.train() keeps dropout ON; run T=50 passes
T_MC = 50
mc_net.train()    # keep dropout active for uncertainty sampling
X_all_mc = X_mc.to(device)
with torch.no_grad():
    samples = torch.stack([mc_net(X_all_mc).squeeze() for _ in range(T_MC)], dim=0)

mean_pred = samples.mean(0).cpu().numpy()
std_pred  = samples.std(0).cpu().numpy()
x_np      = X_mc.squeeze().numpy()

gap_mask = (x_np >= 1) & (x_np <= 2)
unc_gap  = std_pred[gap_mask].mean()
unc_obs  = std_pred[~gap_mask].mean()
print(f"Uncertainty in training gap [1,2]:     {unc_gap:.4f}")
print(f"Uncertainty in observed region:        {unc_obs:.4f}")
print(f"Ratio (gap/observed):                  {unc_gap/max(unc_obs, 1e-9):.2f}x")
print("MC Dropout correctly shows higher uncertainty in unobserved regions.")


## Real-World Example 2: Variational Dropout

In [ ]:
# Variational dropout: learn per-weight dropout probabilities.
# Weights with high alpha (variance/mean ratio) are effectively pruned.

class VariationalDropout(nn.Module):
    # Linear layer with per-weight learned dropout (log-alpha parameterisation).

    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight   = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias     = nn.Parameter(torch.zeros(out_features))
        # Initialise log-alpha low (p_drop near 0 at start)
        self.log_alpha = nn.Parameter(torch.full((out_features, in_features), -3.0))

    @property
    def alpha(self):
        return self.log_alpha.exp().clamp(0, 1)

    def forward(self, x):
        if self.training:
            # Local reparameterisation: sample noise per activation (fast)
            mu  = x @ self.weight.T
            var = (x ** 2) @ (self.weight ** 2 * self.alpha).T
            eps = torch.randn_like(mu)
            return mu + eps * (var.clamp(min=1e-8) ** 0.5) + self.bias
        else:
            # At inference, zero out high-alpha weights (sparsification)
            mask = (self.alpha < 0.99).float()
            return x @ (self.weight * mask).T + self.bias

    def kl_div(self):
        # Approximate KL divergence (encourages sparsity).
        k1, k2, k3 = 0.63576, 1.87320, 1.48695
        C   = -k1
        mdkl = k1 * torch.sigmoid(k2 + k3 * self.log_alpha) - 0.5 * torch.log1p(self.alpha.reciprocal())
        return -(mdkl + C).sum()

vd_net = nn.Sequential(VariationalDropout(10, 64), nn.ReLU(), VariationalDropout(64, 1))
X_vd = torch.randn(400, 10)
y_vd = X_vd[:, :3].sum(1, keepdim=True)
opt_vd = torch.optim.Adam(vd_net.parameters(), lr=1e-3)
BETA_KL = 1e-4

for epoch in range(100):
    vd_net.train()
    opt_vd.zero_grad()
    mse = nn.MSELoss()(vd_net(X_vd), y_vd)
    kl  = sum(m.kl_div() for m in vd_net if isinstance(m, VariationalDropout))
    (mse + BETA_KL * kl).backward()
    opt_vd.step()

vd_net.eval()
pruned = sum((m.alpha >= 0.99).sum().item() for m in vd_net if isinstance(m, VariationalDropout))
total_w = sum(m.alpha.numel() for m in vd_net if isinstance(m, VariationalDropout))
with torch.no_grad():
    mse_final = nn.MSELoss()(vd_net(X_vd), y_vd).item()
print(f"Variational dropout -- MSE: {mse_final:.4f}  sparsity: {pruned}/{total_w} = {pruned/total_w:.2%}")
print("High-alpha weights can be removed post-training for a smaller, faster model.")


## Real-World Example 3: Scheduled Dropout

In [ ]:
# Scheduled dropout: start with high p (strong regularisation early),
# anneal to lower p (let model use capacity later).
# Compare against fixed p=0.5 on the same task.

class ScheduledDropoutMLP(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hid)
        self.fc2 = nn.Linear(d_hid, d_hid)
        self.fc3 = nn.Linear(d_hid, d_out)
        self.dp  = nn.Dropout(p=0.0)   # set dynamically

    def set_p(self, p):
        self.dp.p = p

    def forward(self, x):
        return self.fc3(F.relu(self.dp(self.fc1(x))))

N_sd = 600
X_sd = torch.randn(N_sd, 16)
y_sd = ((X_sd[:,0]**2 + X_sd[:,1] - X_sd[:,2]) > 0).long()
tr_ds_sd = TensorDataset(X_sd[:480].to(device), y_sd[:480].to(device))
ld_sd = DataLoader(tr_ds_sd, batch_size=32, shuffle=True)

EPOCHS_SD = 120
P_START, P_END = 0.7, 0.1

sched_mdl = ScheduledDropoutMLP(16, 64, 2).to(device)
fixed_mdl = build_dropout_mlp(0.5)
opt_sched = torch.optim.Adam(sched_mdl.parameters(), lr=1e-3)
opt_fixed = torch.optim.Adam(fixed_mdl.parameters(), lr=1e-3)
crit_sd   = nn.CrossEntropyLoss()

sched_hist, fixed_hist = [], []
for epoch in range(EPOCHS_SD):
    p_now = P_START - (P_START - P_END) * (epoch / EPOCHS_SD)
    sched_mdl.set_p(p_now)
    sched_mdl.train(); fixed_mdl.train()
    for xb, yb in ld_sd:
        opt_sched.zero_grad()
        crit_sd(sched_mdl(xb), yb).backward()
        opt_sched.step()
        opt_fixed.zero_grad()
        crit_sd(fixed_mdl(xb), yb).backward()
        opt_fixed.step()

    if (epoch + 1) % 20 == 0:
        sched_mdl.eval(); fixed_mdl.eval()
        with torch.no_grad():
            va_s = (sched_mdl(X_sd[480:].to(device)).argmax(1) == y_sd[480:].to(device)).float().mean().item()
            va_f = (fixed_mdl(X_sd[480:].to(device)).argmax(1) == y_sd[480:].to(device)).float().mean().item()
        sched_hist.append(va_s); fixed_hist.append(va_f)
        print(f"Epoch {epoch+1:3d}  p={p_now:.2f}  sched_val={va_s:.4f}  fixed_val={va_f:.4f}")

print(f"Final: scheduled p={P_END} -> {sched_hist[-1]:.4f}  |  fixed p=0.5 -> {fixed_hist[-1]:.4f}")
print("Scheduled dropout: strong early regularisation lets the model specialise later.")
